# Etapa 9 · Dashboard interactivo

Genera un HTML autocontenido desde la copia analítica de Etapa 6, sin modificarla. Ejecutar las celdas en orden desde la raíz del repositorio o `notebooks/`.

Para **reproducir y validar**: Python, pandas, openpyxl, plotly, nbformat y playwright; Chrome/Edge instalado o Chromium de Playwright. Para **abrir el HTML**: únicamente un navegador moderno, sin conexión ni servidor.

Los filtros y gráficos se calculan en JavaScript; las pruebas los comparan con una referencia independiente en pandas. La satisfacción promedio se incorpora solo para el KPI solicitado, junto con mediana y distribución.


In [1]:
from pathlib import Path
import json, hashlib, math, re, sys
import pandas as pd
import plotly
from plotly.offline import get_plotlyjs

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data/analysis/dataset_salud_vulnerabilidad.xlsx").exists())
SOURCE = ROOT / "data/analysis/dataset_salud_vulnerabilidad.xlsx"
OUTPUT = ROOT / "outputs/dashboard/dashboard_salud.html"
PROTECTED = {p: hashlib.sha256(p.read_bytes()).hexdigest()
             for p in ROOT.rglob("*") if p.is_file() and ".git" not in p.parts
             and p not in [OUTPUT, ROOT/"reports/09_dashboard.md", ROOT/"notebooks/09_dashboard.ipynb"]}
df = pd.read_excel(SOURCE)
assert df.shape == (2500, 14)
assert df.ID_Paciente.nunique() == 2500
assert df.Tiempo_Espera_min.isna().sum() == 74
assert df.Satisfaccion.isna().sum() == 72
assert df.Score_Vulnerabilidad.isna().sum() == 74
assert set(df.Score_Vulnerabilidad.dropna()) == {0, 1, 2, 3}
# El score se lee tal como está: no se recalcula ninguna dimensión.
COV = ["Privada", "Pública", "Sin cobertura"]
COND = ["Saludable", "Aguda", "Crónica"]
COLS = ["Region", "Cobertura_Salud", "Condicion_Salud", "Tiempo_Espera_min",
        "Acceso_Medicacion", "Satisfaccion", "Score_Vulnerabilidad"]
# Solo campos necesarios, sin identificadores ni otras características.
records = json.loads(df[COLS].to_json(orient="records", force_ascii=False))
assert len(records) == 2500
assert sum(r["Tiempo_Espera_min"] is None for r in records) == 74
assert sum(r["Satisfaccion"] is None for r in records) == 72
print(f"Fuente leída sin cambios: {df.shape[0]} filas × {df.shape[1]} columnas.")


Fuente leída sin cambios: 2500 filas × 14 columnas.


In [2]:
def clean(x):
    if isinstance(x, dict):
        return {k: clean(v) for k, v in x.items()}
    if isinstance(x, list):
        return [clean(v) for v in x]
    if pd.isna(x):
        return None
    return x.item() if hasattr(x, "item") else x

def reference(part):
    """Referencia independiente con pandas para KPIs y las cinco agregaciones."""
    w = part.Tiempo_Espera_min.dropna()
    s = part.Satisfaccion.dropna()
    a = part[part.Acceso_Medicacion.isin(["Sí", "No"])]
    q = part.dropna(subset=["Score_Vulnerabilidad"])
    pair = part.dropna(subset=["Score_Vulnerabilidad", "Satisfaccion"])
    covers = [c for c in COV if c in set(part.Cobertura_Salud)]
    conditions = [c for c in COND if c in set(part.Condicion_Salud)]
    def percent(n, d):
        return 100*n/d if d else None
    def distribution(frame, col, levels):
        counts = frame[col].value_counts()
        return [{"level": k, "n": int(counts.get(k, 0)), "p": percent(int(counts.get(k, 0)), len(frame))} for k in levels]
    wc, ac, qc, ss = [], [], [], []
    for c in covers:
        cw = part.loc[part.Cobertura_Salud.eq(c), "Tiempo_Espera_min"].dropna()
        wc.append(dict(coverage=c, n=len(cw), mean=cw.mean(), median=cw.median()))
        group = q[q.Cobertura_Salud.eq(c)]
        qc.append(dict(coverage=c, n=len(group), dist=distribution(group, "Score_Vulnerabilidad", range(4))))
    for condition in conditions:
        row = []
        for coverage in covers:
            group = part[part.Cobertura_Salud.eq(coverage) & part.Condicion_Salud.eq(condition)]
            good = group[group.Acceso_Medicacion.isin(["Sí", "No"])]
            yes = int(good.Acceso_Medicacion.eq("Sí").sum())
            row.append(dict(coverage=coverage, condition=condition, total=len(group), n=len(good), yes=yes, p=percent(yes, len(good))))
        ac.append(row)
    for score in range(4):
        group = pair[pair.Score_Vulnerabilidad.eq(score)]
        ss.append(dict(score=score, n=len(group), median=group.Satisfaccion.median(),
                       dist=distribution(group, "Satisfaccion", range(1, 6))))
    return clean(dict(n=len(part), wait=dict(n=len(w), mean=w.mean(), median=w.median()),
        access=dict(n=len(a), yes=int(a.Acceso_Medicacion.eq("Sí").sum()), p=percent(int(a.Acceso_Medicacion.eq("Sí").sum()), len(a))),
        sat=dict(n=len(s), mean=s.mean(), median=s.median()), scoreN=len(q), pairN=len(pair),
        scoreCounts=[int(q.Score_Vulnerabilidad.eq(k).sum()) for k in range(4)], covers=covers, conditions=conditions,
        waitCov=wc, accessCells=ac, satDist=distribution(part.dropna(subset=["Satisfaccion"]), "Satisfaccion", range(1,6)),
        scoreCov=qc, satScore=ss))

global_ref = reference(df)
assert global_ref["n"] == 2500
assert global_ref["wait"]["n"] == 2426 and global_ref["wait"]["median"] == 54
assert abs(global_ref["wait"]["mean"]-64.58) < .005
assert global_ref["access"]["n"] == 2500 and global_ref["access"]["p"] == 67.8
assert global_ref["sat"]["n"] == 2428 and global_ref["sat"]["median"] == 4
assert global_ref["scoreN"] == 2426 and global_ref["pairN"] == 2357
assert global_ref["scoreCounts"] == [904, 937, 471, 114]
for actual, expected in zip(global_ref["waitCov"], [17.60,70.19,106.11]):
    assert abs(actual["mean"]-expected) < .005
for cov, cond, target in [("Privada","Aguda",89.22),("Sin cobertura","Crónica",41.99)]:
    cell = next(c for row in global_ref["accessCells"] for c in row if c["coverage"]==cov and c["condition"]==cond)
    assert abs(cell["p"]-target) < .005
for g, target in zip(global_ref["scoreCov"], [0,3.09,10.90]):
    assert abs(g["dist"][3]["p"]-target) < .005
assert [g["median"] for g in global_ref["satScore"]] == [4,4,3,2]
print(json.dumps(global_ref, ensure_ascii=False, indent=2))


{
  "n": 2500,
  "wait": {
    "n": 2426,
    "mean": 64.580791426216,
    "median": 54.0
  },
  "access": {
    "n": 2500,
    "yes": 1695,
    "p": 67.8
  },
  "sat": {
    "n": 2428,
    "mean": 3.5642504118616145,
    "median": 4.0
  },
  "scoreN": 2426,
  "pairN": 2357,
  "scoreCounts": [
    904,
    937,
    471,
    114
  ],
  "covers": [
    "Privada",
    "Pública",
    "Sin cobertura"
  ],
  "conditions": [
    "Saludable",
    "Aguda",
    "Crónica"
  ],
  "waitCov": [
    {
      "coverage": "Privada",
      "n": 823,
      "mean": 17.601458080194412,
      "median": 18.0
    },
    {
      "coverage": "Pública",
      "n": 777,
      "mean": 70.19176319176319,
      "median": 71.0
    },
    {
      "coverage": "Sin cobertura",
      "n": 826,
      "mean": 106.11138014527845,
      "median": 105.0
    }
  ],
  "accessCells": [
    [
      {
        "coverage": "Privada",
        "condition": "Saludable",
        "total": 272,
        "n": 272,
        "yes": 242,
       

In [3]:
# Plantilla, datos y biblioteca quedan dentro de un único HTML.
TEMPLATE = "<!DOCTYPE html>\n<html lang=\"es\">\n<head>\n<meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n<meta http-equiv=\"Content-Security-Policy\" content=\"default-src 'none'; script-src 'unsafe-inline'; style-src 'unsafe-inline'; img-src data: blob:; font-src data:; connect-src 'none'; object-src 'none'; base-uri 'none'\">\n<title>Salud Pública · Acceso, atención y satisfacción</title>\n<style>\n*{box-sizing:border-box}body{margin:0;background:#f3f6f9;color:#243b4b;font:15px/1.55 \"Segoe UI\",Arial,sans-serif}main{max-width:1320px;margin:auto;padding:36px 28px 28px}\nheader{border-top:5px solid #386d83;padding:20px 0 22px}.eyebrow{font-size:12px;font-weight:700;letter-spacing:.14em;text-transform:uppercase;color:#587184}\nh1{font-size:40px;line-height:1.15;margin:8px 0 5px;font-weight:700;letter-spacing:-1px}.subtitle{font-size:23px;margin:0 0 12px}.muted{color:#607385}.intro{margin:0}.badge{display:inline-block;margin-top:12px;padding:4px 10px;background:#e4edf2;border-radius:5px;font-size:12px}\n.filters{background:#fff;border:1px solid #dce5ed;border-radius:10px;padding:20px;display:grid;grid-template-columns:repeat(3,minmax(0,1fr)) auto;gap:18px;align-items:end}\nlabel{display:block;font-size:13px;font-weight:600;margin-bottom:6px}select,button{font:inherit;min-height:44px;border-radius:5px;border:1px solid #b5c5d1;padding:8px 12px}select{width:100%;background:#fff;color:#243b4b}button{background:#28566c;color:#fff;border-color:#28566c;cursor:pointer}button:hover{background:#1b4054}button:focus-visible,select:focus-visible,summary:focus-visible{outline:3px solid #d5a95d;outline-offset:3px}\n#status{font-size:13px;color:#536a7c;margin:12px 2px 20px}.kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:16px}.kpi{background:#fff;border:1px solid #dce5ed;border-radius:9px;padding:20px 18px;border-top:3px solid #82a5b6}.kpi h2{font-size:11px;letter-spacing:.06em;text-transform:uppercase;margin:0;color:#516b7c}.value{font-size:30px;line-height:1.3;font-weight:700;margin:10px 0;letter-spacing:-.5px}.value.small{font-size:23px}.detail{font-size:12px;color:#536a7c}.ordinal{font-size:12px;color:#536a7c;margin:12px 0 26px;padding-left:2px}\n.section-heading{font-size:21px;margin:26px 0 14px}.grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:20px}.panel{min-width:0;background:#fff;border:1px solid #dce5ed;border-radius:10px;padding:20px}.panel h3{font-size:17px;line-height:1.4;margin:0 0 5px}.panel .meta{font-size:12px;color:#607385;margin:0 0 8px}.chart{height:350px;width:100%}.wide{grid-column:1/-1}.wide .chart{height:350px}.note{font-size:12px;margin:8px 0;color:#536a7c}.index-intro{font-size:14px;color:#536a7c;margin:-5px 0 16px;max-width:950px}\ndetails{font-size:12px;border-top:1px solid #e8eef3;margin-top:12px;padding-top:10px}summary{cursor:pointer;color:#28566c}.table-wrap{overflow:auto;max-height:300px;margin-top:10px}table{border-collapse:collapse;width:100%;font-size:12px;white-space:nowrap}th,td{text-align:right;padding:7px 10px;border-bottom:1px solid #e8eef3}th:first-child,td:first-child{text-align:left}th{background:#f3f6f9;font-weight:600}\n.method{margin-top:24px;padding:20px 24px;border-radius:9px;background:#e8eef3;font-size:13px;color:#4c6375}.method h2{font-size:15px;margin:0 0 8px}.method p{margin:7px 0}footer{font-size:12px;color:#607385;padding:18px 0 0}\n@media(max-width:1000px){.kpis{grid-template-columns:repeat(2,minmax(0,1fr))}.filters{grid-template-columns:repeat(3,minmax(0,1fr))}.filters button{grid-column:1/-1}.grid{grid-template-columns:1fr}.wide{grid-column:auto}}\n@media(max-width:560px){main{padding:18px 12px}h1{font-size:32px}.subtitle{font-size:20px}.filters{grid-template-columns:1fr;gap:12px}.kpis{grid-template-columns:1fr 1fr;gap:10px}.kpi{padding:14px 12px}.value{font-size:25px}.value.small{font-size:21px}.panel{padding:14px 10px}.chart{height:360px}.detail{line-height:1.7}}\n</style>\n<script>__PLOTLY__</script>\n</head>\n<body><main>\n<header><div class=\"eyebrow\">Herramientas de Análisis de Datos · TP1</div><h1>Salud Pública</h1><p class=\"subtitle\">Acceso, atención y satisfacción</p><p class=\"intro muted\">Análisis exploratorio de un dataset ficticio de 2500 registros.</p><span class=\"badge\">Datos ficticios con fines académicos.</span></header>\n<section class=\"filters\" aria-label=\"Filtros combinables\">\n<div><label for=\"Region\">Región</label><select id=\"Region\"><option value=\"\">Todas</option></select></div>\n<div><label for=\"Cobertura_Salud\">Cobertura de salud</label><select id=\"Cobertura_Salud\"><option value=\"\">Todas</option></select></div>\n<div><label for=\"Condicion_Salud\">Condición de salud</label><select id=\"Condicion_Salud\"><option value=\"\">Todas</option></select></div>\n<button type=\"button\" id=\"reset\">Restablecer filtros</button></section>\n<p id=\"status\" role=\"status\" aria-live=\"polite\"></p>\n<section class=\"kpis\" aria-label=\"Indicadores de la selección\">\n<article class=\"kpi\"><h2>Registros</h2><div class=\"value\" id=\"n\">—</div><div class=\"detail\">Coinciden con los filtros actuales</div></article>\n<article class=\"kpi\"><h2>Tiempo promedio de espera</h2><div class=\"value\" id=\"wait\">—</div><div class=\"detail\" id=\"wait-detail\"></div></article>\n<article class=\"kpi\"><h2>Acceso a medicación</h2><div class=\"value\" id=\"access\">—</div><div class=\"detail\" id=\"access-detail\"></div></article>\n<article class=\"kpi\"><h2>Satisfacción</h2><div class=\"value small\" id=\"sat\">—</div><div class=\"detail\" id=\"sat-detail\"></div></article>\n</section>\n<p class=\"ordinal\">La satisfacción se registra en una escala ordinal 1–5. La media se muestra como KPI solicitado y se complementa con mediana y distribución.</p>\n<h2 class=\"section-heading\">Acceso y experiencia de atención</h2>\n<div class=\"grid\">\n<section class=\"panel\"><h3>Tiempo promedio de espera por cobertura</h3><p class=\"meta\" id=\"note1\"></p><div id=\"chart1\" class=\"chart\" role=\"img\" aria-label=\"Barras de espera promedio por cobertura\"></div><details><summary>Ver valores y denominadores</summary><div class=\"table-wrap\" id=\"table1\"></div></details></section>\n<section class=\"panel\"><h3>Acceso a medicación según cobertura y condición de salud</h3><p class=\"meta\" id=\"note2\"></p><div id=\"chart2\" class=\"chart\" role=\"img\" aria-label=\"Porcentaje de acceso a medicación por cobertura y condición\"></div><details><summary>Ver valores y denominadores</summary><div class=\"table-wrap\" id=\"table2\"></div></details></section>\n<section class=\"panel wide\"><h3>Distribución de satisfacción</h3><p class=\"meta\" id=\"note3\"></p><div id=\"chart3\" class=\"chart\" role=\"img\" aria-label=\"Distribución ordenada de satisfacción del uno al cinco\"></div><details><summary>Ver valores y denominadores</summary><div class=\"table-wrap\" id=\"table3\"></div></details></section>\n</div>\n<h2 class=\"section-heading\">Índice operacional de vulnerabilidad en el acceso y la atención</h2>\n<p class=\"index-intro\">El score representa cantidad de dimensiones operacionales observadas; no es una escala clínica. Se conserva la definición del ejercicio: condición crónica, falta de acceso a medicación y espera de al menos 99 minutos.</p>\n<div class=\"grid\">\n<section class=\"panel\"><h3>Acumulación de dimensiones según cobertura</h3><p class=\"meta\" id=\"note4\"></p><div id=\"chart4\" class=\"chart\" role=\"img\" aria-label=\"Distribución de scores cero a tres dentro de cada cobertura\"></div><p class=\"note\">Cada cobertura usa solo sus registros con score observable. Un score sin información no equivale a cero.</p><details><summary>Ver valores y denominadores</summary><div class=\"table-wrap\" id=\"table4\"></div></details></section>\n<section class=\"panel\"><h3>Satisfacción según acumulación de dimensiones</h3><p class=\"meta\" id=\"note5\"></p><div id=\"chart5\" class=\"chart\" role=\"img\" aria-label=\"Distribución de satisfacción dentro de cada score\"></div><p class=\"note\">Asociación descriptiva; no implica causalidad ni validación clínica del índice.</p><details><summary>Ver valores y denominadores</summary><div class=\"table-wrap\" id=\"table5\"></div></details></section>\n</div>\n<section class=\"method\"><h2>Notas metodológicas</h2><p>Dataset ficticio con 2500 registros originales, conservados íntegramente. Los indicadores descriptivos utilizan casos disponibles: los NA no se imputan y cada vista informa su N válido dentro de la selección.</p><p>Satisfacción es ordinal, de 1 a 5; su promedio se acompaña de mediana y distribución. El índice operacional se creó para este ejercicio: score 0–3 = cantidad de dimensiones. El umbral de espera es fijo, 99 minutos (Q3 del dataset), y no se recalcula al filtrar.</p><p>Las asociaciones no implican causalidad. Los segmentos pequeños requieren cautela y estos datos no representan a la población real.</p></section>\n<footer>TP1 · Salud Pública · Exploración descriptiva</footer>\n<noscript>Para utilizar los filtros, habilite JavaScript en el navegador.</noscript>\n</main>\n<script type=\"application/json\" id=\"records\">__DATA__</script>\n<script id=\"dashboard-code\">__APP__</script>\n</body></html>"
APP_JS = "\"use strict\";\nconst DATA = JSON.parse(document.getElementById(\"records\").textContent);\nconst COV = [\"Privada\",\"Pública\",\"Sin cobertura\"];\nconst COND = [\"Saludable\",\"Aguda\",\"Crónica\"];\nconst LEVELS = [1,2,3,4,5], SCORES = [0,1,2,3];\nconst satColors = [\"#924434\",\"#bf7250\",\"#b6a46c\",\"#548d94\",\"#24536a\"];\nconst scoreColors = [\"#dde6ef\",\"#9cb8d2\",\"#537fa3\",\"#254965\"];\nconst covColors = [\"#386d83\",\"#548d94\",\"#8a718e\"];\nconst valid = x => typeof x === \"number\" && Number.isFinite(x);\nconst vals = (rows,k) => rows.map(r=>r[k]).filter(valid);\nconst mean = a => a.length ? a.reduce((s,v)=>s+v,0)/a.length : null;\nconst median = a => {if(!a.length)return null; const b=[...a].sort((x,y)=>x-y),m=Math.floor(b.length/2);return b.length%2?b[m]:(b[m-1]+b[m])/2;};\nconst pct = (n,d) => d ? 100*n/d : null;\nconst fmt = (n,d=2) => n===null ? \"Sin datos\" : n.toLocaleString(\"es-AR\",{minimumFractionDigits:d,maximumFractionDigits:d});\nconst count = n => fmt(n,0);\nconst selectRows = (data,filters) => data.filter(r=>Object.entries(filters).every(([k,v])=>!v||r[k]===v));\nfunction aggregate(rows) {\n const w=vals(rows,\"Tiempo_Espera_min\"), s=vals(rows,\"Satisfaccion\");\n const a=rows.filter(r=>r.Acceso_Medicacion===\"Sí\"||r.Acceso_Medicacion===\"No\");\n const q=rows.filter(r=>valid(r.Score_Vulnerabilidad));\n const pair=q.filter(r=>valid(r.Satisfaccion));\n const covers=COV.filter(c=>rows.some(r=>r.Cobertura_Salud===c));\n const conditions=COND.filter(c=>rows.some(r=>r.Condicion_Salud===c));\n const dist=(rs,key,levels)=>levels.map(level=>({level,n:rs.filter(r=>r[key]===level).length,p:pct(rs.filter(r=>r[key]===level).length,rs.length)}));\n return {\n  n:rows.length, wait:{n:w.length,mean:mean(w),median:median(w)},\n  access:{n:a.length,yes:a.filter(r=>r.Acceso_Medicacion===\"Sí\").length,p:pct(a.filter(r=>r.Acceso_Medicacion===\"Sí\").length,a.length)},\n  sat:{n:s.length,mean:mean(s),median:median(s)}, scoreN:q.length,pairN:pair.length,\n  scoreCounts:SCORES.map(k=>q.filter(r=>r.Score_Vulnerabilidad===k).length),\n  covers,conditions,\n  waitCov:covers.map(c=>{const x=vals(rows.filter(r=>r.Cobertura_Salud===c),\"Tiempo_Espera_min\");return {coverage:c,n:x.length,mean:mean(x),median:median(x)};}),\n  accessCells:conditions.map(condition=>covers.map(coverage=>{\n    const group=rows.filter(r=>r.Cobertura_Salud===coverage&&r.Condicion_Salud===condition);\n    const good=group.filter(r=>r.Acceso_Medicacion===\"Sí\"||r.Acceso_Medicacion===\"No\");\n    return {coverage,condition,total:group.length,n:good.length,yes:good.filter(r=>r.Acceso_Medicacion===\"Sí\").length,p:pct(good.filter(r=>r.Acceso_Medicacion===\"Sí\").length,good.length)};\n  })),\n  satDist:dist(rows.filter(r=>valid(r.Satisfaccion)),\"Satisfaccion\",LEVELS),\n  scoreCov:covers.map(coverage=>{const g=q.filter(r=>r.Cobertura_Salud===coverage);return {coverage,n:g.length,dist:dist(g,\"Score_Vulnerabilidad\",SCORES)};}),\n  satScore:SCORES.map(score=>{const g=pair.filter(r=>r.Score_Vulnerabilidad===score);return {score,n:g.length,median:median(vals(g,\"Satisfaccion\")),dist:dist(g,\"Satisfaccion\",LEVELS)};})\n };\n}\nconst config={responsive:true,displaylogo:false,displayModeBar:false,scrollZoom:false};\nfunction layout(yTitle,percent=false){\n return {font:{family:\"Segoe UI, Arial, sans-serif\",size:13,color:\"#33485b\"},paper_bgcolor:\"#fff\",plot_bgcolor:\"#fff\",\n margin:{l:56,r:20,t:18,b:65},autosize:true,separators:\",.\",hoverlabel:{font:{size:13}},\n xaxis:{fixedrange:true,automargin:true,type:\"category\"},\n yaxis:{title:{text:yTitle},fixedrange:true,gridcolor:\"#e8eef3\",zeroline:false,automargin:true,...(percent?{range:[0,100],ticksuffix:\"%\"}:{rangemode:\"tozero\"})},\n legend:{orientation:\"h\",x:0,y:-.24,traceorder:\"normal\",itemclick:false,itemdoubleclick:false},\n bargap:.36,showlegend:false};\n}\nfunction chart(id,traces,lay,empty) {\n if(empty) {traces=[];lay.annotations=[{text:\"Sin datos para la selección\",xref:\"paper\",yref:\"paper\",x:.5,y:.5,showarrow:false,font:{size:16,color:\"#617184\"}}];}\n return Plotly.react(id,traces,lay,config);\n}\nfunction details(id,headers,rows){\n const host=document.getElementById(id);host.replaceChildren();\n const table=document.createElement(\"table\"),head=document.createElement(\"thead\"),hr=document.createElement(\"tr\");\n headers.forEach(h=>{const th=document.createElement(\"th\");th.textContent=h;hr.append(th);});head.append(hr);table.append(head);\n const body=document.createElement(\"tbody\");\n rows.forEach(r=>{const tr=document.createElement(\"tr\");r.forEach(v=>{const td=document.createElement(\"td\");td.textContent=v;tr.append(td);});body.append(tr);});\n table.append(body);host.append(table);\n}\nasync function render(filters) {\n const rows=selectRows(DATA,filters),a=aggregate(rows);\n document.getElementById(\"n\").textContent=count(a.n);\n document.getElementById(\"wait\").textContent=a.wait.mean===null?\"Sin datos\":fmt(a.wait.mean)+\" min\";\n document.getElementById(\"wait-detail\").textContent=\"Mediana: \"+fmt(a.wait.median,1)+\" min · N válido: \"+count(a.wait.n);\n document.getElementById(\"access\").textContent=a.access.p===null?\"Sin datos\":fmt(a.access.p)+\"%\";\n document.getElementById(\"access-detail\").textContent=\"N válido: \"+count(a.access.n);\n document.getElementById(\"sat\").textContent=a.sat.mean===null?\"Sin datos\":\"Promedio: \"+fmt(a.sat.mean)+\" / 5\";\n document.getElementById(\"sat-detail\").textContent=\"Mediana: \"+fmt(a.sat.median,1)+\" / 5 · N válido: \"+count(a.sat.n);\n document.getElementById(\"status\").textContent=a.n?count(a.n)+\" registros · \"+(Object.values(filters).filter(Boolean).join(\" / \")||\"Todas las regiones, coberturas y condiciones\"):\"Sin datos para la selección\";\n const note=(id,n)=>document.getElementById(id).textContent=\"N válido: \"+count(n)+\" · Sin información para esta vista: \"+count(a.n-n);\n note(\"note1\",a.wait.n);note(\"note2\",a.access.n);note(\"note3\",a.sat.n);note(\"note4\",a.scoreN);note(\"note5\",a.pairN);\n const tasks=[];\n const w=a.waitCov;\n tasks.push(chart(\"chart1\",[{type:\"bar\",x:w.map(x=>x.coverage),y:w.map(x=>x.mean),marker:{color:w.map(x=>covColors[COV.indexOf(x.coverage)])},\n text:w.map(x=>x.mean===null?\"Sin datos\":fmt(x.mean)),textposition:\"auto\",\n customdata:w.map(x=>[x.median,x.n]),hovertemplate:\"%{x}<br>Promedio: %{y:.2f} min<br>Mediana: %{customdata[0]} min<br>N válido: %{customdata[1]}<extra></extra>\"}],layout(\"Minutos\"),!a.wait.n));\n const cells=a.accessCells;\n const hl=layout(\"Condición de salud\");hl.margin.r=65;hl.yaxis={type:\"category\",fixedrange:true,automargin:true,autorange:\"reversed\"};\n tasks.push(chart(\"chart2\",[{type:\"heatmap\",x:a.covers,y:a.conditions,z:cells.map(r=>r.map(c=>c.p)),zmin:0,zmax:100,\n colorscale:[[0,\"#edf3f7\"],[.5,\"#82acb9\"],[1,\"#24536a\"]],colorbar:{title:{text:\"% Sí\"},ticksuffix:\"%\",thickness:12},\n text:cells.map(r=>r.map(c=>c.p===null?\"Sin datos\":fmt(c.p)+\"%\")),texttemplate:\"%{text}\",hoverongaps:false,\n customdata:cells.map(r=>r.map(c=>[c.total,c.n,c.yes])),\n hovertemplate:\"Cobertura: %{x}<br>Condición: %{y}<br>Acceso Sí: %{z:.2f}%<br>N del grupo: %{customdata[0]}<br>N válido: %{customdata[1]}<br>Sí: %{customdata[2]}<extra></extra>\"}],hl,!a.access.n));\n tasks.push(chart(\"chart3\",[{type:\"bar\",x:LEVELS.map(String),y:a.satDist.map(x=>x.p),marker:{color:satColors},\n text:a.satDist.map(x=>x.p===null?\"\":fmt(x.p)+\"%\"),textposition:\"auto\",\n customdata:a.satDist.map(x=>[x.n,a.sat.n]),hovertemplate:\"Satisfacción %{x}<br>Cantidad: %{customdata[0]}<br>Porcentaje: %{y:.2f}%<br>N válido: %{customdata[1]}<extra></extra>\"}],layout(\"Casos con satisfacción (%)\",true),!a.sat.n));\n const ql=layout(\"Casos con score (%)\",true);ql.barmode=\"stack\";ql.showlegend=true;\n tasks.push(chart(\"chart4\",SCORES.map((s,i)=>({type:\"bar\",name:\"Score \"+s,x:a.covers,y:a.scoreCov.map(g=>g.dist[i].p),\n marker:{color:scoreColors[i],line:{color:\"#fff\",width:1}},text:a.scoreCov.map(g=>g.dist[i].p>=7?fmt(g.dist[i].p,1)+\"%\":\"\"),textposition:\"inside\",\n customdata:a.scoreCov.map(g=>[g.dist[i].n,g.n]),\n hovertemplate:\"%{x}<br>Score \"+s+\"<br>Cantidad: %{customdata[0]}<br>Porcentaje: %{y:.2f}%<br>N válido de cobertura: %{customdata[1]}<extra></extra>\"})),ql,!a.scoreN));\n const sl=layout(\"Casos con ambas variables (%)\",true);sl.barmode=\"stack\";sl.showlegend=true;\n tasks.push(chart(\"chart5\",LEVELS.map((level,i)=>({type:\"bar\",name:\"Satisfacción \"+level,x:SCORES.map(s=>\"Score \"+s),y:a.satScore.map(g=>g.dist[i].p),\n marker:{color:satColors[i],line:{color:\"#fff\",width:1}},text:a.satScore.map(g=>g.dist[i].p>=7?fmt(g.dist[i].p,1)+\"%\":\"\"),textposition:\"inside\",\n customdata:a.satScore.map(g=>[g.dist[i].n,g.n,g.median]),\n hovertemplate:\"%{x}<br>Satisfacción \"+level+\"<br>Cantidad: %{customdata[0]}<br>Porcentaje: %{y:.2f}%<br>N del score: %{customdata[1]}<br>Mediana del score: %{customdata[2]}<extra></extra>\"})),sl,!a.pairN));\n details(\"table1\",[\"Cobertura\",\"Promedio (min)\",\"Mediana (min)\",\"N válido\"],w.map(g=>[g.coverage,fmt(g.mean),fmt(g.median,1),g.n]));\n details(\"table2\",[\"Cobertura\",\"Condición\",\"Acceso Sí (%)\",\"Sí\",\"N válido\",\"N grupo\"],cells.flat().map(g=>[g.coverage,g.condition,fmt(g.p),g.yes,g.n,g.total]));\n details(\"table3\",[\"Satisfacción\",\"Cantidad\",\"Porcentaje\",\"N válido\"],a.satDist.map(g=>[g.level,g.n,fmt(g.p),a.sat.n]));\n details(\"table4\",[\"Cobertura\",\"Score\",\"Cantidad\",\"Porcentaje\",\"N válido\"],a.scoreCov.flatMap(g=>g.dist.map(x=>[g.coverage,x.level,x.n,fmt(x.p),g.n])));\n details(\"table5\",[\"Score\",\"Satisfacción\",\"Cantidad\",\"Porcentaje\",\"N del score\"],a.satScore.flatMap(g=>g.dist.map(x=>[g.score,x.level,x.n,fmt(x.p),g.n])));\n await Promise.all(tasks);\n window.dashboardState={filters,summary:a};\n return a;\n}\nconst selectors=[\"Region\",\"Cobertura_Salud\",\"Condicion_Salud\"];\nselectors.forEach(k=>{\n const el=document.getElementById(k);\n const values=k===\"Cobertura_Salud\"?COV:k===\"Condicion_Salud\"?COND:[...new Set(DATA.map(r=>r[k]))].sort((a,b)=>a.localeCompare(b,\"es\"));\n values.forEach(v=>{const o=document.createElement(\"option\");o.value=v;o.textContent=v;el.append(o);});\n el.addEventListener(\"change\",()=>{window.dashboardReady=render(currentFilters());});\n});\nfunction currentFilters(){return Object.fromEntries(selectors.map(k=>[k,document.getElementById(k).value]));}\ndocument.getElementById(\"reset\").addEventListener(\"click\",()=>{selectors.forEach(k=>document.getElementById(k).value=\"\");window.dashboardReady=render(currentFilters());});\nwindow.dashboardAPI={aggregate,selectRows,render};\nwindow.dashboardReady=render(currentFilters());\n"
payload = json.dumps(records, ensure_ascii=False, allow_nan=False, separators=(",", ":")).replace("<", "\\u003c")
html = TEMPLATE.replace("__PLOTLY__", get_plotlyjs()).replace("__DATA__", payload).replace("__APP__", APP_JS)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT.write_text(html, encoding="utf-8")
print(f"HTML generado: {OUTPUT.relative_to(ROOT)} ({OUTPUT.stat().st_size:,} bytes)")


HTML generado: outputs\dashboard\dashboard_salud.html (5,278,162 bytes)


In [4]:
from playwright.sync_api import sync_playwright
from html.parser import HTMLParser
import tempfile
import os

class ResourceParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.external = []
    def handle_starttag(self, tag, attrs):
        for key, value in attrs:
            if key in ["src", "href", "action", "poster"] and value and not value.startswith(("data:", "blob:", "#")):
                self.external.append((tag, key, value))
parser = ResourceParser()
parser.feed(html)
assert not parser.external
assert 'id="records"' in html and 'Plotly' in html
assert len(json.loads(re.search(r'<script type="application/json" id="records">(.*?)</script>', html, re.S).group(1))) == 2500
assert ".xlsx" not in html and ".ipynb" not in html
# Las URL internas del bundle no equivalen a recursos cargados por este dashboard.
tokens = ["http://", "https://", "cdn.plot.ly", "file://", "localhost", "127.0.0.1"]
url_counts = {token: html.count(token) for token in tokens}
assert all(token not in TEMPLATE and token not in APP_JS for token in tokens)
assert 'connect-src \'none\'' in html
assert OUTPUT.exists() and OUTPUT.stat().st_size > 0

def compare(actual, expected, path=""):
    if isinstance(expected, dict):
        assert set(actual) == set(expected), path
        for k, v in expected.items():
            compare(actual[k], v, path+"."+k)
    elif isinstance(expected, list):
        assert len(actual) == len(expected), path
        for i, v in enumerate(expected):
            compare(actual[i], v, path+f"[{i}]")
    elif isinstance(expected, (int, float)):
        assert actual is not None and math.isclose(actual, expected, rel_tol=1e-10, abs_tol=1e-9), (path, actual, expected)
    else:
        assert actual == expected, (path, actual, expected)

def check_totals(a):
    groups = [dict(n=a["sat"]["n"], dist=a["satDist"])] + a["scoreCov"] + a["satScore"]
    for group in groups:
        assert sum(x["n"] for x in group["dist"]) == group["n"]
        if group["n"]:
            assert math.isclose(sum(x["p"] for x in group["dist"]), 100, abs_tol=1e-9)
        else:
            assert all(x["p"] is None for x in group["dist"])

cases = [
    ("A", {"Region":"CABA"}),
    ("B", {"Cobertura_Salud":"Pública"}),
    ("C", {"Condicion_Salud":"Crónica"}),
    ("D", {"Region":"CABA","Cobertura_Salud":"Pública","Condicion_Salud":"Crónica"})
]
browser_results = []
errors, network = [], []
def validate_in_browser():
    with sync_playwright() as pw:
        # Chrome o Edge instalados; alternativa: navegador administrado por Playwright.
        candidates = [Path(os.environ.get("PROGRAMFILES",""))/"Google/Chrome/Application/chrome.exe",
                      Path(os.environ.get("PROGRAMFILES(X86)",""))/"Microsoft/Edge/Application/msedge.exe"]
        executable = next((str(p) for p in candidates if p.is_file()), None)
        browser = pw.chromium.launch(headless=True, **({"executable_path":executable} if executable else {}))
        context = browser.new_context(offline=True, viewport={"width":1440,"height":1100})
        page = context.new_page()
        page.on("pageerror", lambda e: errors.append(str(e)))
        page.on("request", lambda r: network.append(r.url) if r.url.startswith(("http:", "https:")) else None)
        page.goto(OUTPUT.as_uri())
        page.evaluate("() => window.dashboardReady")
        compare(page.evaluate("() => window.dashboardState.summary"), global_ref)
        assert page.locator(".js-plotly-plot").count() == 5
        def check_rendered(a):
            traces = page.evaluate("() => [1,2,3,4,5].map(i=>document.getElementById('chart'+i).data)")
            compare(traces[0][0]["y"], [g["mean"] for g in a["waitCov"]])
            compare(traces[1][0]["z"], [[c["p"] for c in row] for row in a["accessCells"]])
            compare(traces[2][0]["y"], [g["p"] for g in a["satDist"]])
            for k in range(4):
                compare(traces[3][k]["y"], [g["dist"][k]["p"] for g in a["scoreCov"]])
            for k in range(5):
                compare(traces[4][k]["y"], [g["dist"][k]["p"] for g in a["satScore"]])
            for id_, value in [("n",a["n"]),("wait",a["wait"]["mean"]),("access",a["access"]["p"]),("sat",a["sat"]["mean"])]:
                digits = 0 if id_=="n" else 2
                formatted = page.evaluate("([x,d])=>x.toLocaleString('es-AR',{minimumFractionDigits:d,maximumFractionDigits:d})",[value,digits])
                assert formatted in page.locator("#"+id_).inner_text()
        check_rendered(global_ref)
        for name, filters in cases:
            page.locator("#reset").click()
            page.evaluate("() => window.dashboardReady")
            for key, value in filters.items():
                page.select_option("#"+key, value)
                page.evaluate("() => window.dashboardReady")
            part = df.copy()
            for key, value in filters.items():
                part = part[part[key].eq(value)]
            assert all(part[k].eq(v).all() for k,v in filters.items())
            # Identidad de filas seleccionadas comprobada sin incluir ID_Paciente en el HTML.
            expected_rows = json.loads(part[COLS].to_json(orient="records", force_ascii=False))
            got_rows = page.evaluate("(f)=>window.dashboardAPI.selectRows(JSON.parse(document.getElementById('records').textContent),f)", filters)
            compare(got_rows, expected_rows)
            expected = reference(part)
            actual = page.evaluate("() => window.dashboardState.summary")
            compare(actual, expected)
            check_totals(actual)
            check_rendered(actual)
            browser_results.append(dict(test=name, filters=filters, n=actual["n"],
                wait_n=actual["wait"]["n"], access_n=actual["access"]["n"],
                sat_n=actual["sat"]["n"], score_n=actual["scoreN"], pair_n=actual["pairN"], result="APROBADO"))
        # Selección vacía, sin alterar datos ni ofrecer una categoría artificial al usuario.
        page.evaluate("() => window.dashboardAPI.render({Region:'__sin_coincidencias__'})")
        assert page.locator("#status").inner_text() == "Sin datos para la selección"
        assert page.evaluate("() => [1,2,3,4,5].every(i=>document.getElementById('chart'+i).data.length===0)")
        assert page.locator("#wait").inner_text() == "Sin datos"
        # Casos límite adicionales: grupos existentes sin observables y conteo de score 0.
        probe = [dict(Region="CABA",Cobertura_Salud="Pública",Condicion_Salud="Crónica",
                      Tiempo_Espera_min=None,Acceso_Medicacion=None,Satisfaccion=None,Score_Vulnerabilidad=None)]
        zero = page.evaluate("(rows)=>window.dashboardAPI.aggregate(rows)",probe)
        assert zero["n"]==1 and zero["wait"]["mean"] is None and zero["sat"]["mean"] is None
        check_totals(zero)
        page.locator("#reset").click()
        page.evaluate("() => window.dashboardReady")
        compare(page.evaluate("() => window.dashboardState.summary"),global_ref)
        assert all(page.input_value("#"+k)=="" for k in ["Region","Cobertura_Salud","Condicion_Salud"])
        page.locator("#chart1").screenshot(path=str(Path(tempfile.gettempdir())/"tp1_dashboard_chart.png"))
        page.screenshot(path=str(Path(tempfile.gettempdir())/"tp1_dashboard_desktop.png"), full_page=True)
        # Hover real, para comprobar que Plotly conserva información interactiva.
        page.locator("#chart1 .bars .point").first.hover(force=True)
        assert "Promedio:" in page.locator("#chart1 .hoverlayer").text_content()
        page.set_viewport_size({"width":390,"height":844})
        page.wait_for_timeout(400)
        assert page.evaluate("() => document.documentElement.scrollWidth <= window.innerWidth")
        page.screenshot(path=str(Path(tempfile.gettempdir())/"tp1_dashboard_mobile.png"), full_page=True)
        assert not errors, errors
        assert not network, network
        context.close()
        browser.close()
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=1) as pool:
    pool.submit(validate_in_browser).result()
print(json.dumps({"filters":browser_results,"url_counts":url_counts,"network_requests":network,
                  "javascript_errors":errors,"reset":"APROBADO","empty":"APROBADO","offline":"APROBADO"},ensure_ascii=False,indent=2))


{
  "filters": [
    {
      "test": "A",
      "filters": {
        "Region": "CABA"
      },
      "n": 498,
      "wait_n": 479,
      "access_n": 498,
      "sat_n": 480,
      "score_n": 479,
      "pair_n": 463,
      "result": "APROBADO"
    },
    {
      "test": "B",
      "filters": {
        "Cobertura_Salud": "Pública"
      },
      "n": 806,
      "wait_n": 777,
      "access_n": 806,
      "sat_n": 780,
      "score_n": 777,
      "pair_n": 752,
      "result": "APROBADO"
    },
    {
      "test": "C",
      "filters": {
        "Condicion_Salud": "Crónica"
      },
      "n": 843,
      "wait_n": 824,
      "access_n": 843,
      "sat_n": 820,
      "score_n": 824,
      "pair_n": 802,
      "result": "APROBADO"
    },
    {
      "test": "D",
      "filters": {
        "Region": "CABA",
        "Cobertura_Salud": "Pública",
        "Condicion_Salud": "Crónica"
      },
      "n": 41,
      "wait_n": 41,
      "access_n": 41,
      "sat_n": 38,
      "score_n": 41,
   

In [5]:
def md_table(headers, rows, numeric=()):
    rows = [[str(v) for v in row] for row in rows]
    widths = [max(len(h), *(len(r[i]) for r in rows), 3) for i, h in enumerate(headers)]
    def line(row):
        return "| " + " | ".join(v.rjust(widths[i]) if i in numeric else v.ljust(widths[i]) for i,v in enumerate(row)) + " |"
    sep = "| " + " | ".join("-"*(w-1)+":" if i in numeric else "-"*w for i,w in enumerate(widths)) + " |"
    return "\n".join([line(headers),sep,*[line(r) for r in rows]])

report = """# Etapa 9 - Dashboard interactivo

## 1. Objetivo

Comunicar tiempos de espera, acceso a medicación, satisfacción e Índice operacional de vulnerabilidad en el acceso y la atención mediante una página interactiva. Se leyeron los informes de Etapas 1–8 antes de construirla.

La consigna de Etapa 9 resuelve las decisiones pendientes de selección: cinco gráficos, cuatro KPIs, satisfacción promedio acompañada de mediana y distribución, y exclusión del modelado. No se modifican retrospectivamente los informes anteriores ni se elaboran recomendaciones o presentación.

## 2. Arquitectura

Python lee el Excel y genera un único HTML UTF-8 con Plotly JS, estilos, datos JSON y lógica JavaScript embebidos. Al cambiar un filtro se seleccionan registros, se recalculan KPIs y agregaciones y se actualizan las cinco vistas mediante Plotly.react(). No se precalculan todas las combinaciones.

La documentación oficial respalda la [exportación autocontenida de Plotly](https://plotly.com/python/interactive-html-export/) y la [actualización mediante Plotly.react](https://plotly.com/javascript/plotlyjs-function-reference/). Estos enlaces son referencias de este informe, no dependencias del HTML.

El [notebook reproducible](../notebooks/09_dashboard.ipynb) contiene carga, controles globales, plantilla completa, lógica JavaScript, pruebas independientes en pandas y navegador, y generación de este informe. Para reproducir y validar se requieren Python, pandas, openpyxl, plotly, nbformat, playwright y un navegador compatible. Chrome o Edge instalado se detectan automáticamente; alternativamente puede instalarse Chromium de Playwright. La prueba se ejecuta en un hilo separado para convivir con el bucle de eventos de Jupyter. Ninguna de estas herramientas es requerida por quien abre el HTML.

## 3. Fuente de datos

Se utiliza exclusivamente data/analysis/dataset_salud_vulnerabilidad.xlsx: 2500 filas × 14 columnas. Se conserva intacto el archivo y se leen sus scores preexistentes sin volver a calcular dimensiones, umbrales ni reglas.

El HTML incluye los 2500 registros con siete campos necesarios: Region, Cobertura_Salud, Condicion_Salud, Tiempo_Espera_min, Acceso_Medicacion, Satisfaccion y Score_Vulnerabilidad. No incorpora ID_Paciente ni campos ajenos a las vistas. Los NA se serializan como null; no se convierten en cero.

## 4. Filtros

Selectores combinables de Region, Cobertura_Salud y Condicion_Salud, todos inicialmente en “Todas”, con botón “Restablecer filtros”. Se usa intersección de condiciones: cada registro debe cumplir todos los filtros activos. Una cobertura específica conserva la pregunta del gráfico de espera y muestra una única barra.

Los KPIs, cinco gráficos, denominadores, notas de casos no disponibles y tablas desplegables se actualizan conjuntamente. Una selección vacía muestra “Sin datos para la selección”, sin divisiones por cero ni errores JavaScript. Los selectores usan categorías reales; el caso vacío se comprueba mediante una llamada de prueba al mismo renderizador.

## 5. KPIs

"""
report += md_table(["KPI","Resultado global","Denominador / complemento"],[
    ["Registros","2500","Todos los registros seleccionados"],
    ["Tiempo promedio de espera",f'{global_ref["wait"]["mean"]:.2f} min',"N=2426; mediana=54 min"],
    ["Acceso a medicación",f'{global_ref["access"]["p"]:.2f}%',"Sí / acceso observable; N=2500"],
    ["Satisfacción promedio",f'{global_ref["sat"]["mean"]:.2f} / 5',"N=2428; mediana=4 / 5"]
])
report += f"""

**Satisfacción promedio global calculada para este KPI:** {global_ref["sat"]["mean"]:.12f} / 5. Se muestra redondeada a 3,56 / 5. Esta incorporación está autorizada expresamente en Etapa 9; no sustituye la mediana ni presupone intervalos iguales entre categorías. La nota ordinal se mantiene visible inmediatamente debajo de las tarjetas.

## 6. Visualizaciones

"""
report += md_table(["Vista","Tipo","Denominador reactivo","Información accesible"],[
    ["Espera por cobertura","Barras","Espera observable en cada cobertura","Promedio, mediana y N válido"],
    ["Acceso por cobertura y condición","Heatmap","Acceso observable en cada combinación","Porcentaje Sí, cantidad Sí, N válido y N del grupo"],
    ["Distribución de satisfacción","Barras ordenadas 1–5","Satisfacción observable del filtro","Nivel, cantidad, porcentaje y N válido"],
    ["Acumulación por cobertura","Barras apiladas al 100%; score 0–3","Score observable en cada cobertura","Score, cantidad, porcentaje y N de cobertura"],
    ["Satisfacción según acumulación","Barras apiladas al 100%; satisfacción 1–5","Ambas variables observables dentro de cada score","Score, nivel, cantidad, porcentaje, N y mediana del score"]
])
report += """

Se utilizan paletas consistentes para satisfacción y score, leyendas ordenadas, porcentajes y tooltips. Las tablas desplegables brindan una alternativa al color y permiten consultar categorías con frecuencia cero. Las celdas sin denominador válido se muestran sin porcentaje, no como 0%. Los porcentajes se recalculan desde conteos, no desde cifras globales redondeadas.

La página tiene fondo claro, tarjetas, separación entre bloques y adaptación a escritorio y móvil. Se verificó a 1440 y 390 píxeles de ancho, sin desbordamiento horizontal de la página. Las leyendas no ocultan segmentos al pulsarlas: se preserva la lectura de distribuciones completas al 100%.

## 7. Tratamiento de NA

Los 74 NA de espera y 72 de satisfacción permanecen ausentes. El score conserva 74 NA. Cada KPI excluye únicamente ausencias de su propia variable; acceso usa respuestas Sí/No observables. El gráfico conjunto exige score y satisfacción, con N global=2357 y 143 registros sin alguna de ambas variables.

Un NA no se interpreta como score 0, satisfacción 0 o falta de acceso. Las medias, medianas, porcentajes y N válidos se calculan sobre el subconjunto actual. Cuando no hay observables, se presenta “Sin datos”; una categoría con observables pero cero casos del nivel sí conserva porcentaje cero.

## 8. Índice operacional

Se mantiene el nombre **Índice operacional de vulnerabilidad en el acceso y la atención**. Score 0–3 significa cantidad de dimensiones operacionales: condición crónica, falta de acceso a medicación y espera ≥99 minutos. El umbral es el Q3 original y permanece fijo al filtrar.

No se crea una clasificación binaria ni categorías de gravedad. El índice no es una escala clínica; la asociación con satisfacción no implica causalidad ni validación clínica. La cobertura solo segmenta; no se incorpora a la fórmula.

## 9. Validación sin filtros

La referencia independiente en pandas reproduce las cifras anteriores con tolerancia de 0.005 para valores publicados con dos decimales. Se compara además toda la estructura de agregaciones JavaScript con pandas, con tolerancia numérica de 1e-9 absoluta o 1e-10 relativa.

"""
report += md_table(["Control","Resultado","Estado"],[
    ["Registros",2500,"APROBADO"],
    ["Espera: N / media / mediana",f'2426 / {global_ref["wait"]["mean"]:.8f} / 54',"APROBADO"],
    ["Acceso: N / Sí (%)","2500 / 67.80","APROBADO"],
    ["Satisfacción: N / media / mediana",f'2428 / {global_ref["sat"]["mean"]:.12f} / 4',"APROBADO"],
    ["Score: N / frecuencias 0,1,2,3","2426 / 904, 937, 471, 114","APROBADO"],
    ["Score y satisfacción: N","2357","APROBADO"],
    ["Espera media: Privada / Pública / Sin cobertura","17.60 / 70.19 / 106.11 min","APROBADO"],
    ["Acceso: Privada-Aguda / Sin cobertura-Crónica","89.22% / 41.99%","APROBADO"],
    ["Score 3: Privada / Pública / Sin cobertura","0.00% / 3.09% / 10.90%","APROBADO"],
    ["Mediana satisfacción por score 0,1,2,3","4 / 4 / 3 / 2","APROBADO"]
])
report += """

## 10. Validación de filtros

Se accionaron los selectores reales en Chrome/Chromium headless, con red deshabilitada, y se compararon las filas seleccionadas con pandas en contenido y orden. Se verificaron todos los KPIs, N válidos, cinco agregaciones y los datos efectivamente entregados a los cinco gráficos. Ningún registro fuera de la selección participó.

"""
report += md_table(["Prueba","Selección","Registros","N espera","N acceso","N satisfacción","N score","N conjunto","Estado"],[
    [r["test"]," / ".join(r["filters"].values()),r["n"],r["wait_n"],r["access_n"],r["sat_n"],r["score_n"],r["pair_n"],r["result"]]
    for r in browser_results],numeric=(2,3,4,5,6,7))
report += """

Cada distribución con denominador positivo suma aproximadamente 100% antes del redondeo. Para grupos sin observables, los porcentajes son null. Se verificaron también reset a 2500 registros, estado vacío, grupo de prueba con todas las variables descriptivas ausentes, tooltips y diseño responsive. El grupo artificial se usa únicamente en la función de prueba; no se agrega al HTML ni al dataset.

## 11. Portabilidad

"""
report += md_table(["Control","Resultado"],[
    ["HTML", "outputs/dashboard/dashboard_salud.html"],
    ["Tamaño",f"{OUTPUT.stat().st_size} bytes ({OUTPUT.stat().st_size/1024/1024:.2f} MiB)"],
    ["Plotly JS","Embebido íntegramente"],
    ["Datos","JSON embebido: 2500 registros × 7 campos necesarios"],
    ["Recursos externos de HTML","0"],
    ["Solicitudes HTTP/HTTPS al abrir y filtrar","0"],
    ["Errores JavaScript","0"],
    ["Apertura","Archivo local con navegador offline, sin servidor"],
    ["Dependencia de Excel o notebook para abrir","Ninguna"],
    ["Reset y cinco gráficos","APROBADOS"],
    ["Política de red","connect-src 'none'; recursos externos bloqueados"]
])
report += "\n\n**Búsqueda explícita de referencias:**\n\n"
report += md_table(["Texto buscado","Apariciones en HTML","Interpretación"],[
    [token,n,"Solo dentro del bundle Plotly; no usado como dependencia por estas vistas" if n else "Ausente"]
    for token,n in url_counts.items()],numeric=(1,))
report += """

Los literales HTTP/HTTPS y cdn.plot.ly pertenecen al código y referencias internas de la biblioteca completa, incluidas funciones ajenas a estas vistas; no todos son texto documental. Los literales file:// también pertenecen al bundle y no apuntan a archivos del proyecto. No hay atributos externos src/href ni referencias a XLSX o notebook. La plantilla propia, los datos y la lógica del dashboard no requieren ninguna de esas direcciones.

Además de la inspección estática se probó apertura directa por URI local, con el contexto del navegador offline, filtros y reset. No se registró ninguna solicitud HTTP/HTTPS. La política de contenido bloquea conexiones de red; las cinco vistas usan barras y heatmap, sin mapas, servicios externos ni fuentes remotas. Por ello la presencia de literales internos del bundle no representa una dependencia funcional.

Para compartir: entregar únicamente el HTML y abrirlo con doble clic en un navegador moderno. La vista previa de GitHub o de algunos clientes de correo puede no ejecutar JavaScript: descargar el archivo y abrirlo en el navegador.

## 12. Limitaciones

Los datos son ficticios y las asociaciones descriptivas no establecen causalidad ni representan la población real. Filtrar puede producir grupos pequeños; se muestran sus N, sin asignar umbrales ni etiquetas nuevas. Los valores faltantes tienen mecanismo desconocido. El promedio de satisfacción es un KPI solicitado sobre una variable ordinal y debe leerse con sus complementos.

El HTML incorpora la información necesaria para explorar los 2500 registros; es una copia estática de la fuente al generarlo. Cambios posteriores en el Excel requieren regeneración con el notebook. La biblioteca completa aumenta el tamaño a favor de portabilidad offline.

Las únicas advertencias técnicas son los literales de URL internos del bundle y la necesidad de descargar el HTML para ejecutarlo fuera de visores que bloquean scripts. No hay errores funcionales en las pruebas realizadas. No se incluyeron modelado, pruebas inferenciales, recomendaciones ni presentación.

"""
assert all(p.exists() and hashlib.sha256(p.read_bytes()).hexdigest()==digest for p,digest in PROTECTED.items())
report += f"**Integridad:** todos los archivos anteriores conservaron su SHA-256. Fuente analítica: {hashlib.sha256(SOURCE.read_bytes()).hexdigest()}.\n\n"
report += f"Entorno de generación: Python {sys.version.split()[0]}, pandas {pd.__version__}, Plotly Python {plotly.__version__}.\n"
(ROOT/"reports/09_dashboard.md").write_text(report,encoding="utf-8")
print(f"Informe generado. Integridad de {len(PROTECTED)} archivos anteriores verificada.")


Informe generado. Integridad de 47 archivos anteriores verificada.
